# DistilBERT Fine-Tuning — FinancialPhraseBank

Fine-tunes `distilbert-base-uncased` for 3-class sentiment (Negative / Neutral / Positive)  
and saves the result to `./model/` so the Dockerfile can copy it into the Lambda container.

**Run this notebook once** on a machine with a GPU (Colab, Kaggle, SageMaker) or locally on CPU if you're patient.  
After it finishes, run `convert_to_onnx.py` to produce the quantized model the Lambda uses.

In [98]:
# Install — safe to run in Colab/Kaggle; no-op if already installed locally
!pip install transformers datasets evaluate accelerate scikit-learn -q

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)



[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [99]:
import os
import re
import sys
import numpy as np
from pathlib import Path

import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
)
import evaluate

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

Using device: cpu


In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
MODEL_CHECKPOINT = "distilbert-base-uncased"
OUTPUT_DIR = "./model" # matches Dockerfile COPY model/ and app.py MODEL_DIR default
CHECKPOINT_DIR = "./checkpoints" # intermediate training checkpoints — safe to delete after

# sentences_allagree = only sentences where every human annotator agreed on the label
# Cleaner signal than 75agree/66agree/50agree; 2264 sentences total.
DATASET_CONFIG = "sentences_allagree"

# Label mapping written into config.json — app.py reads this at inference time
ID2LABEL = {0: "Negative", 1: "Neutral", 2: "Positive"}
LABEL2ID = {v: k for k, v in ID2LABEL.items()}

MAX_LENGTH = 128    # DistilBERT max is 512; financial sentences are short — 128 is plenty
BATCH_SIZE = 16
LEARNING_RATE = 2e-4
EPOCHS = 2
SEED = 42

In [ ]:
# Dataset
raw = load_dataset("zeroshot/twitter-financial-news-sentiment", trust_remote_code=True)
raw = raw.class_encode_column("label")

def clean_financial_text(text):
    # 1. Remove cashtags (e.g., $BYND, $CCL, $RCL)
    text = re.sub(r'\$\w+', '', text)
    
    # 2. Remove URLs (e.g., https://t.co/...)
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    
    # 3. Remove the dash separator often used after tickers
    text = text.replace('-', '')
    
    # 4. Clean up extra whitespace/newlines
    text = " ".join(text.split())
    
    return text

# Apply this to your dataset
# Assuming 'dataset' is your Hugging Face dataset object
cleaned_dataset = raw.map(lambda x: {"text": clean_financial_text(x["text"])})

split = cleaned_dataset["train"].train_test_split(test_size=0.2, seed=SEED, stratify_by_column="label")
train_ds = split["train"]
val_ds = split["test"]

print(f"Train: {len(train_ds)} | Val: {len(val_ds)}")
print(f"Label distribution (train): {dict(zip(*np.unique(train_ds['label'], return_counts=True)))}")

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'zeroshot/twitter-financial-news-sentiment' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Train: 7634 | Val: 1909
Label distribution (train): {np.int64(0): np.int64(1154), np.int64(1): np.int64(1538), np.int64(2): np.int64(4942)}


In [ ]:
# Tokenise
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)

train_tok = train_ds.map(tokenize, batched=True, remove_columns=["text"])
val_tok   = val_ds.map(tokenize,   batched=True, remove_columns=["text"])

train_tok.set_format("torch")
val_tok.set_format("torch")

In [ ]:
# Model: id2label / label2id are baked into config.json so app.py reads them correctly
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=len(ID2LABEL),
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)
print(f"Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Parameters: 67.0M


In [ ]:
# Metrics
accuracy_metric = evaluate.load("accuracy")
f1_metric       = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        **accuracy_metric.compute(predictions=preds, references=labels),
        **f1_metric.compute(predictions=preds, references=labels, average="macro"),
    }

In [ ]:
# Training arguments
training_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    warmup_ratio=0.1,           # linear warmup over first 10% of steps
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    fp16=(DEVICE == "cuda"),    # mixed precision on GPU only; CPU doesn't support fp16
    seed=SEED,
    logging_steps=25,
    report_to="none",           # set to "wandb" if you want experiment tracking
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

/var/folders/8l/td67j21x0z11wzxv4pp00nyh0000gn/T/ipykernel_85787/270525041.py:21: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
# Train
trainer.train()

  0%|          | 0/956 [00:00<?, ?it/s]/Users/krishnashenoy/Desktop/datascience/portfilio_sentiment/lib/python3.13/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
  1%|▏         | 14/956 [00:04<04:02,  3.89it/s]

KeyboardInterrupt: 

In [ ]:
# Evaluate model
results = trainer.evaluate()
print(f"\nValidation accuracy : {results['eval_accuracy']:.4f}")
print(f"Validation F1 macro : {results['eval_f1']:.4f}")

# Quick sanity check — predict a few sentences manually
from transformers import pipeline as hf_pipeline
pipe = hf_pipeline("text-classification", model=model, tokenizer=tokenizer, device=-1, top_k=None)
samples = [
    "The company reported record profits this quarter.",
    "Revenue declined sharply due to weak consumer demand.",
    "The board approved the annual dividend payment.",
]
for s in samples:
    top = max(pipe(s)[0], key=lambda x: x["score"])
    print(f"{top['label']:8s} ({top['score']:.2%})  →  {s[:60]}")

/Users/krishnashenoy/Desktop/datascience/portfilio_sentiment/lib/python3.13/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
100%|██████████| 60/60 [00:02<00:00, 23.27it/s]


Validation accuracy : 0.8502
Validation F1 macro : 0.8038
Neutral  (90.66%)  →  The company reported record profits this quarter.
Negative (97.07%)  →  Revenue declined sharply due to weak consumer demand.
Positive (99.35%)  →  The board approved the annual dividend payment.


In [ ]:
# Save: OUTPUT_DIR = "./model" — matches the Dockerfile COPY and app.py MODEL_DIR default
Path(OUTPUT_DIR).mkdir(exist_ok=True)
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

saved = [f.name for f in Path(OUTPUT_DIR).iterdir()]
print(f"Saved to {OUTPUT_DIR}/")
print("  ", "\n  ".join(sorted(saved)))

Saved to ./model/
   config.json
  model.safetensors
  special_tokens_map.json
  tokenizer.json
  tokenizer_config.json
  vocab.txt


In [ ]:
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    import shutil
    zip_path = shutil.make_archive("model", "zip", ".", "model")
    print(f"Created {zip_path} — downloading…")
    from google.colab import files
    files.download(zip_path)
    print("Unzip into the project root, then run: python convert_to_onnx.py")
else:
    print("Running locally — model is already at ./model")
    print("Next step: python convert_to_onnx.py --model_dir ./model --output_dir ./model/onnx")

Running locally — model is already at ./model
Next step: python convert_to_onnx.py --model_dir ./model --output_dir ./model/onnx
